## 🦀 Torch
---

In [2]:
:dep tch

In [3]:
use tch::{Kind, Tensor, Device};
use tch::nn::{Module, Path};
use tch::nn::init::{Init, NormalOrUniform, FanInOut, NonLinearity};

In [4]:
let dev = Device::cuda_if_available();
dev

Cpu

---
### Vector Basics:

In [5]:
let a = Tensor::from_slice(&[1.0, 2.0, 3.0]);
let b = Tensor::from_slice(&[4.0, 5.0, 6.0]);
let c = &a + &b;
println!("a + b = {:?}", c);
let d = a.dot(&b);
println!("a · b = {:?}", d);

a + b = [5.0, 7.0, 9.0]
a · b = [32]


---
### Matrix Basics:

In [6]:
let A = Tensor::rand([3, 4], (tch::Kind::Float, dev));
let B = Tensor::rand([4, 6], (tch::Kind::Float, dev));
let C = A.matmul(&B);
println!("A · B = ");
C.print()

A · B = 
 0.6086  0.3417  0.7639  0.7485  0.8289  0.7204
 1.0551  0.7787  1.1926  1.3947  1.2697  0.9301
 1.1800  0.4866  1.3441  1.6069  1.3432  1.2280
[ CPUFloatType{3,6} ]


()

---
### Network Basics:

In [7]:
#[derive(Debug)]
struct CustomLinear {
    weight: Tensor,
}

In [8]:
impl Module for CustomLinear {
    fn forward(&self, xs: &Tensor) -> Tensor {
        xs.matmul(&self.weight.tr())
    }
}

In [9]:
fn custom_linear(p: &Path, in_features: i64, out_features: i64) -> CustomLinear {
    let weight = p.var("weight", &[out_features, in_features], Init::Kaiming {
        dist: NormalOrUniform::Uniform,
        fan: FanInOut::FanIn,
        non_linearity: NonLinearity::ReLU,
    });
    CustomLinear { weight }
}

In [10]:
{
    let vs = tch::nn::VarStore::new(dev);
    let root = &vs.root();
    let linear = custom_linear(root, 4, 2);
    let x = Tensor::randn(&[3, 4], (Kind::Float, Device::Cpu));
    let y = linear.forward(&x);
    println!("Output:\n{:?}", y);
};

Output:
Tensor[[3, 2], Float]
